# Sentence window

## Get started

<img src="sentence_window.png">

## Prepare the data

我们使用 Langchain WebBaseLoader 从博客源加载文档，并通过 RecursiveCharacterTextSplitter 将其拆分为多个片段。

In [1]:
import os

CUSTOM_CACHE = r'F:\Teewon\Milvue\models'
os.environ['HF_HOME'] = CUSTOM_CACHE
os.environ['HF_HUB_CACHE'] = os.path.join(CUSTOM_CACHE, 'hub')
os.environ['TRANSFORMERS_CACHE'] = os.path.join(CUSTOM_CACHE, 'transformers')
os.environ['TORCH_HOME'] = CUSTOM_CACHE

In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Create a WebBaseLoader instance to load documents from web sources
loader = WebBaseLoader(
    web_path=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content","post-title","post-header")
        ),# 只解析 HTML 中符合特定条件的部分
    )
)

# Load documents from web sources using the loader
documents=loader.load()

# Initialize a RecursiveCharacterTextSplitter for splitting text into chunks
text_spliiter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=0)

# Split the documents into chunks using the text_splitter
docs=text_spliiter.split_documents(documents)

# Inspect
docs[1]

C:\Users\Administrator\AppData\Local\Temp\ipykernel_1556\2095123501.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Short-term memory: I would consider all the in-context learning (See Prompt Engineering) as utilizing short-term memory of the model to learn.\nLong-term memory: This provides the agent with the capability to retain and recall (infinite) information over extended periods, often by leveraging an external vector store and fast retrieval.\n\n\nTool use\n\nThe agent learns to call external APIs for extra information that is missing from the model weights (often hard to change after pre-training), including current information, code execution capability, access to proprietary information sources and more.\n\n\n\n\n\nOverview of a LLM-powered autonomous agent system.')

我们使用 write_wider_window 函数将更宽的窗口文本信息添加到每个片段中。可以看到文档的元数据包含一个 wider_text 字段，即更宽的窗口文本信息。

In [4]:
from rag_utils.sentence_window import write_wider_window

write_wider_window(docs,documents[0],offset=500)

print(len(docs))
docs[1]

60


Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 979, 'end_index': 1635, 'winder_text': 't System Overview#\nIn a LLM-powered autonomous agent system, LLM functions as the agent’s brain, complemented by several key components:\n\nPlanning\n\nSubgoal and decomposition: The agent breaks down large tasks into smaller, manageable subgoals, enabling efficient handling of complex tasks.\nReflection and refinement: The agent can do self-criticism and self-reflection over past actions, learn from mistakes and refine them for future steps, thereby improving the quality of final results.\n\n\nMemory\n\nShort-term memory: I would consider all the in-context learning (See Prompt Engineering) as utilizing short-term memory of the model to learn.\nLong-term memory: This provides the agent with the capability to retain and recall (infinite) information over extended periods, often by leveraging an external vector store and fast retrieval.\n\n\nTool u

In [5]:
from rag_utils.vanilla import vectorstore,format_docs,rag_prompt,llm

vectorstore.add_documents(docs)
retriever = vectorstore.as_retriever()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Define the vanilla RAG chain.

In [6]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.prompts import PromptTemplate

vanilla_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
)

Define the sentence window chain.

In [7]:
from rag_utils.sentence_window import format_docs_with_winder_window

sentence_window_chain=(
    {"context":retriever| format_docs_with_winder_window,"question":RunnablePassthrough()}
    |rag_prompt
    | llm
    | StrOutputParser()
)

## Test the chain

In [8]:
# ANN算法有哪些不同类型？
query = "what are the different types of ANN algorithms?"

vanilla_result=vanilla_rag_chain.invoke(query)
sentence_window_result=sentence_window_chain.invoke(query)
print(f"[vanilla_result]:\n {vanilla_result}\n\n[sentence_window_result]:\n {sentence_window_result}")

[vanilla_result]:
 The context describes four common ANN algorithms for fast maximum inner-product search (MIPS):

- **FAISS**: Uses vector quantization by partitioning the vector space into clusters, first searching coarse cluster candidates, then refining within clusters.
- **ScaNN**: Uses anisotropic vector quantization to better preserve inner products between queries and quantized data points.
- **LSH**: Uses locality-sensitive hashing so similar inputs map to the same buckets with high probability.
- **ANNOY**: Uses random projection trees, searching all trees toward the half closest to the query and aggregating results.

[sentence_window_result]:
 The context lists several common ANN algorithms for fast maximum inner-product search (MIPS):

- **LSH (Locality-Sensitive Hashing)**: Uses hash functions so similar inputs map to the same buckets with high probability, with far fewer buckets than inputs.
- **ANNOY (Approximate Nearest Neighbors Oh Yeah)**: Uses random projection trees

## 回答质量对比
| 方法	                  |回答内容	|评价|
|----------------------|---|---|
| Vanilla RAG	         |列出了 4 种 ANN 算法（FAISS、ScaNN、LSH、ANNOY），并给出了每种算法的简要原理描述。	|✅ 回答准确、清晰，但 遗漏了 HNSW，在覆盖完整性上有所欠缺。
| Sentence Window RAG	 |列出了 5 种 ANN 算法（LSH、ANNOY、HNSW、FAISS、ScaNN），对每种算法的描述更加详细，并提及了 recall@10 的对比（虽未给具体数值）。	|✅ 覆盖更全面（包含了 HNSW），描述更丰富，信息量更大，且提到了对比基准，整体内容更完整、专业。|